# 07 · CNN Architectures: Classification and Detection
**Use a GPU if you can:** Runtime ➜ Change runtime type ➜ GPU. Only one section (plain vs ResNet on CIFAR-10) needs real training time; it is marked **GPU recommended**.

**Part A — classification architectures**, in the order they were invented. For each: the **one key idea**, a **small example**, and whether Keras has it **pretrained**.

| | | |
|---|---|---|
| A1 · Timeline at a glance | A5 · VGG: stack 3×3 filters | A9 · MobileNet: depthwise-separable |
| A2 · LeNet-5: the blueprint | A6 · GoogLeNet / Inception: parallel branches | A10 · EfficientNet: scale everything together |
| A3 · AlexNet: deep + ReLU + dropout + GPU | A7 · ResNet: skip connections | A11 · ConvNeXt: a modern CNN |
| A4 · Network-in-Network: 1×1 convolution + GAP | A8 · DenseNet: connect every layer | A12 · Six pretrained models, head to head |

**Part B — detection architectures:** B1 sliding windows · B2 two-stage (R-CNN ➜ Faster R-CNN, anchors) · B3 one-stage (SSD, YOLO) · B4 RetinaNet and the focal loss · B5 two pretrained detectors, head to head · B6 which detector when?

In [1]:
# Colab: RetinaNet comes from KerasHub, Faster R-CNN from TensorFlow Hub (needs setuptools' pkg_resources)
!pip install -q keras-hub tensorflow-hub "setuptools<81"

In [ ]:
import time
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

tf.get_logger().setLevel("ERROR")
keras.utils.set_random_seed(42)
np.set_printoptions(precision=3, suppress=True, linewidth=120)

GPU = bool(tf.config.list_physical_devices("GPU"))
TRAIN_SIZE = 50000 if GPU else 10000                              # CIFAR-10 training photos used
EPOCHS = 10                                                       # per CIFAR network in Part 9
print("GPU:", "yes" if GPU else "none — using", TRAIN_SIZE, "CIFAR training photos (training is much slower on a CPU)")

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train, y_train = x_train[:TRAIN_SIZE].astype("float32") / 255, y_train[:TRAIN_SIZE].ravel()
x_test, y_test = x_test.astype("float32") / 255, y_test.ravel()
(mx_train, my_train), (mx_test, my_test) = keras.datasets.mnist.load_data()
mx_train, mx_test = mx_train[..., None].astype("float32") / 255, mx_test[..., None].astype("float32") / 255
LOSS = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print("CIFAR-10 train", x_train.shape, "test", x_test.shape, "| MNIST train", mx_train.shape)

GPU: yes 50000 CIFAR training photos (training is much slower on a CPU)
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1504s 9us/step
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
CIFAR-10 train (50000, 32, 32, 3) test (10000, 32, 32, 3) | MNIST train (60000, 28, 28, 1)


---
# A1 · Timeline at a glance
The **ImageNet** challenge (1.2 million training photos, 1,000 classes) drove most of these ideas. The score: **top-5 error** — how often the right class is *not* among the model's 5 best guesses (a trained person: about 5%).

| year | network | layers | weights | ImageNet top-5 error | key idea |
|---|---|---|---|---|---|
| 1998 | LeNet-5 | 5 | 0.06 M | — (digits) | conv ➜ pool ➜ dense |
| 2012 | **AlexNet** | 8 | 61 M | 15.3% | deep CNN on GPUs, ReLU, dropout |
| 2013 | Network-in-Network | — | — | — | 1×1 convolution, global average pooling |
| 2014 | **VGG16** | 16 | 138 M | 7.3% | only 3×3 filters, stacked |
| 2014 | **GoogLeNet** | 22 | 7 M | 6.7% | parallel branches (Inception), 1×1 bottlenecks |
| 2015 | **ResNet-152** | 152 | 60 M | 3.6% | skip connections |
| 2017 | DenseNet-121 | 121 | 8 M | — | every layer connected to every later one |
| 2017 | MobileNet | 28 | 4 M | — | depthwise-separable convolution (phones) |
| 2019 | EfficientNet-B0…B7 | — | 5–66 M | — | scale depth, width and image size together |
| 2022 | ConvNeXt | — | 28 M+ | — | a modern CNN redesigned with ideas from transformers |

(Competition scores as published; "—" = not an ILSVRC entry, compared later by top-1 accuracy instead.) Error fell from 15.3% to 3.6% in three years — mostly by going **deeper** and finding ways to **train** deep networks.

---
# A2 · LeNet-5 (Yann LeCun, 1998) — the first famous CNN
Built to read handwritten digits on bank cheques: conv ➜ pool ➜ conv ➜ pool ➜ three dense layers, with tanh and **average** pooling (ReLU and max pooling came later). Every CNN since follows this blueprint. (The LeNet-5 notebook builds it step by step; here we only reuse it.)

In [ ]:
def lenet(input_shape):
    return keras.Sequential([
        keras.Input(input_shape),
        layers.Conv2D(6, 5, padding="same", activation="tanh"), layers.AveragePooling2D(2),     # 28×28×6 ➜ 14×14×6
        layers.Conv2D(16, 5, activation="tanh"), layers.AveragePooling2D(2),                    # 10×10×16 ➜ 5×5×16
        layers.Flatten(),
        layers.Dense(120, activation="tanh"), layers.Dense(84, activation="tanh"), layers.Dense(10),
    ])

lenet_mnist = lenet((28, 28, 1))
lenet_mnist.summary()
lenet_mnist.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
lenet_mnist.fit(mx_train, my_train, batch_size=64, epochs=3, verbose=0)
print(f"LeNet-5 on MNIST digits     : {100 * lenet_mnist.evaluate(mx_test, my_test, verbose=0)[1]:.2f}% with {lenet_mnist.count_params():,} weights")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

LeNet-5 on MNIST digits     : 98.02% with 61,706 weights


The same LeNet on CIFAR-10 colour photos (32×32×3), trained for 10 epochs:

In [ ]:
lenet_cifar = lenet((32, 32, 3))
lenet_cifar.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
h = lenet_cifar.fit(x_train, y_train, batch_size=64, epochs=10, verbose=0).history
print(f"LeNet-5 on CIFAR-10 photos  : train {100 * h['accuracy'][-1]:.1f}% | test {100 * lenet_cifar.evaluate(x_test, y_test, verbose=0)[1]:.1f}% "
      f"with {lenet_cifar.count_params():,} weights")

**Our run:** LeNet-5 reads digits almost perfectly (**98.0%**) — but on colour photos it reaches only **45.1%** test (52.4% train), against 70% for the deeper, regularised CNN of the CIFAR notebook. Two small conv layers with 6 and 16 filters are enough for black-and-white strokes; photos of animals and vehicles, with backgrounds, need **more filters and more layers**. Everything after LeNet is about going deeper without the problems that brings.

*(Numbers in the text are from our run: 10,000 CIFAR training photos on a laptop CPU. On a GPU with all 50,000 photos every CIFAR model scores higher, and CPU runs differ slightly from run to run — compare the pattern, not the exact numbers.)*

---
# A3 · AlexNet (2012): the network that started the deep-learning boom
LeNet's blueprint, but **much bigger**, trained on 1.2 million ImageNet photos on two **GPUs**. It cut the ImageNet error from 26% to 15.3% in one year. Four ideas made it work:
1. **depth and size**: 5 conv layers + 3 dense layers, 61 million weights
2. **ReLU** instead of tanh — trains much faster
3. **dropout** in the dense layers — against overfitting (CIFAR notebook)
4. **data augmentation** — random crops and flips (CIFAR notebook)

Build it with the Functional API (input 227×227×3) and count where the weights are:
```
227×227×3 ─► conv 11×11, 96, stride 4 ─► max pool ─► conv 5×5, 256 ─► max pool ─► conv 3×3, 384 ─► conv 3×3, 384 ─► conv 3×3, 256 ─► max pool
          ─► flatten (6×6×256 = 9,216) ─► dense 4096 ─► dropout ─► dense 4096 ─► dropout ─► dense 1000
```

In [ ]:
inputs = keras.Input(shape=(227, 227, 3))
x = layers.Conv2D(96, 11, strides=4, activation="relu", name="conv1")(inputs)          # 55×55×96
x = layers.MaxPooling2D(3, strides=2)(x)                                               # 27×27×96
x = layers.Conv2D(256, 5, padding="same", activation="relu", name="conv2")(x)          # 27×27×256
x = layers.MaxPooling2D(3, strides=2)(x)                                               # 13×13×256
x = layers.Conv2D(384, 3, padding="same", activation="relu", name="conv3")(x)
x = layers.Conv2D(384, 3, padding="same", activation="relu", name="conv4")(x)
x = layers.Conv2D(256, 3, padding="same", activation="relu", name="conv5")(x)          # 13×13×256
x = layers.MaxPooling2D(3, strides=2)(x)                                               # 6×6×256
x = layers.Flatten()(x)                                                                # 9,216
x = layers.Dense(4096, activation="relu", name="fc6")(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(4096, activation="relu", name="fc7")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1000, name="fc8")(x)
alexnet = keras.Model(inputs, outputs, name="AlexNet")

conv_weights = sum(l.count_params() for l in alexnet.layers if l.name.startswith("conv"))
dense_weights = sum(l.count_params() for l in alexnet.layers if l.name.startswith("fc"))
print(f"{'layer':>6} | {'output':>15} | {'weights':>11}")
print("-" * 40)
for l in alexnet.layers:
    if l.count_params():
        print(f"{l.name:>6} | {str(l.output.shape[1:]):>15} | {l.count_params():>11,}")
print(f"\ntotal {alexnet.count_params():,} weights: conv layers {conv_weights:,} ({100 * conv_weights / alexnet.count_params():.0f}%), "
      f"dense layers {dense_weights:,} ({100 * dense_weights / alexnet.count_params():.0f}%)")

The 5 conv layers do the seeing with only a few % of the weights; the 3 dense layers hold almost all of them (fc6 alone connects 9,216 inputs to 4,096 neurons). Later networks attack exactly this: global average pooling (A4) instead of huge dense layers.

AlexNet is **not** in `keras.applications` (it was overtaken quickly), so there are no pretrained weights here — but its ideas are in every later network.

### ReLU vs tanh: why AlexNet switched
The same small CNN on our CIFAR photos, 3 epochs, once with tanh and once with ReLU:

In [ ]:
def small_cnn(activation):
    inputs = keras.Input((32, 32, 3))
    x = inputs
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation=activation)(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation=activation)(x)
    return keras.Model(inputs, layers.Dense(10)(x))

print(f"{'activation':>10} | {'training accuracy after epochs 1 / 2 / 3':>40} | {'test':>6}")
print("-" * 66)
for activation in ["tanh", "relu"]:
    keras.utils.set_random_seed(42)
    model = small_cnn(activation)
    model.compile(optimizer=keras.optimizers.SGD(0.01, momentum=0.9), loss=LOSS, metrics=["accuracy"])   # plain SGD + momentum, as in 2012
    h = model.fit(x_train, y_train, batch_size=64, epochs=3, verbose=0).history
    print(f"{activation:>10} | {' / '.join(f'{100 * a:5.1f}%' for a in h['accuracy']):>40} | {100 * model.evaluate(x_test, y_test, verbose=0)[1]:5.1f}%")

__RELU__

---
# A4 · Network-in-Network (2013): the 1×1 convolution and global average pooling
Two small ideas from one paper — both are now in almost every CNN:
1. a **1×1 convolution** = a tiny dense layer applied at every pixel (it mixes the channels of one pixel, does not look at neighbours)
2. **global average pooling** at the end, instead of AlexNet's huge dense layers

## The 1×1 convolution: mixing and shrinking channels
A 1×1 filter looks at **one pixel** but all its channels: it is a small dense layer applied at every position. Used to cut the number of channels cheaply (e.g. 256 ➜ 64) before an expensive 3×3 layer.

In [ ]:
x = np.random.default_rng(0).normal(0, 1, (1, 8, 8, 256)).astype("float32")
one_by_one = layers.Conv2D(64, 1)
out = one_by_one(x)
W, b = one_by_one.get_weights()                                   # W: (1, 1, 256, 64)
by_hand = x[0, 3, 5] @ W[0, 0] + b                                # one pixel's 256 values × a 256×64 matrix
print("input", x.shape, "➜ output", tuple(out.shape))
print("pixel (3, 5) by hand == Conv2D 1×1 output?", np.allclose(by_hand, out[0, 3, 5], atol=1e-5))
three_by_three = layers.Conv2D(64, 3)
three_by_three.build(x.shape)
print(f"weights: 1×1 conv 256➜64 = {one_by_one.count_params():,} | 3×3 conv 256➜64 = {three_by_three.count_params():,}")

## NiN's second idea: global average pooling (GAP) instead of Flatten + big dense layers
At the end of a CNN, **Flatten** turns e.g. 8×8×64 maps into 4,096 numbers (a big dense layer follows). **GAP** keeps just the **average of each map**: 64 numbers.

In [ ]:
maps = np.random.default_rng(1).random((2, 8, 8, 64)).astype("float32")
print("GAP by hand (mean over height and width) == layer?", np.allclose(maps.mean(axis=(1, 2)), layers.GlobalAveragePooling2D()(maps)))
for name, head in [("Flatten + Dense(10)", [layers.Flatten(), layers.Dense(10)]),
                   ("GAP + Dense(10)", [layers.GlobalAveragePooling2D(), layers.Dense(10)])]:
    model = keras.Sequential([keras.Input((8, 8, 64)), *head])
    print(f"{name:>20}: {model.layers[0].output.shape[-1]:>5} numbers into the classifier | {model.count_params():>6,} weights")

---
# A5 · VGG (2014): only small 3×3 filters, stacked
Two stacked 3×3 layers see a 5×5 area; three see 7×7 (receptive field, convolution-basics notebook) — with **fewer weights** and more ReLUs in between.

In [ ]:
def count(*stack):
    model = keras.Sequential([keras.Input((32, 32, 64)), *stack])
    return model.count_params()

C = 64
pairs = [("one 5×5", count(layers.Conv2D(C, 5, padding="same")), "two 3×3", count(layers.Conv2D(C, 3, padding="same"), layers.Conv2D(C, 3, padding="same"))),
         ("one 7×7", count(layers.Conv2D(C, 7, padding="same")), "three 3×3", count(*[layers.Conv2D(C, 3, padding="same") for _ in range(3)]))]
for big, big_w, small, small_w in pairs:
    print(f"{big} conv ({C}➜{C} channels): {big_w:>7,} weights | {small}: {small_w:>7,} weights ({100 * (1 - small_w / big_w):.0f}% fewer)")

impulse = np.zeros((1, 15, 15, 1), dtype="float32")
impulse[0, 7, 7, 0] = 1
for n_layers in [1, 2, 3]:
    stack = keras.Sequential([keras.Input((15, 15, 1))] + [layers.Conv2D(1, 3, padding="same", kernel_initializer="ones") for _ in range(n_layers)])
    seen = int((stack(impulse)[0, :, :, 0].numpy().sum(axis=1) != 0).sum())
    print(f"{n_layers} stacked 3×3 layer(s): one input pixel reaches a {seen}×{seen} area")

---
# A6 · GoogLeNet / Inception (2014): several filter sizes side by side
Which filter size is best — 1×1, 3×3 or 5×5? Inception's answer: **all of them**, in parallel branches, and stack the results as channels. Small details are caught by the small filters, bigger shapes by the big ones.
```
                 ┌─ 1×1 conv (64) ───────────────────┐
input 28×28×192 ─┼─ 1×1 (96) ➜ 3×3 conv (128) ───────┼─► concatenate ➜ 28×28×256
                 ├─ 1×1 (16) ➜ 5×5 conv (32) ────────┤
                 └─ 3×3 max pool ➜ 1×1 (32) ─────────┘
```
These are the real sizes of GoogLeNet's first Inception block. The 1×1 layers **before** the 3×3 and 5×5 shrink the channels first (A4). How much do they save?

In [ ]:
def inception_block(shrink):
    inputs = keras.Input((28, 28, 192))
    b1 = layers.Conv2D(64, 1, padding="same", activation="relu")(inputs)
    b3 = layers.Conv2D(96, 1, activation="relu")(inputs) if shrink else inputs
    b3 = layers.Conv2D(128, 3, padding="same", activation="relu")(b3)
    b5 = layers.Conv2D(16, 1, activation="relu")(inputs) if shrink else inputs
    b5 = layers.Conv2D(32, 5, padding="same", activation="relu")(b5)
    bp = layers.MaxPooling2D(3, strides=1, padding="same")(inputs)
    bp = layers.Conv2D(32, 1, activation="relu")(bp)
    return keras.Model(inputs, layers.Concatenate()([b1, b3, b5, bp]))

x = np.random.default_rng(1).random((1, 28, 28, 192)).astype("float32")
for shrink in [False, True]:
    block = inception_block(shrink)
    print(f"{'with' if shrink else 'without':>7} 1×1 shrinking: output {tuple(block(x).shape)} | weights {block.count_params():>7,}")

**Our run:** the same output (28×28×256) with **393,472** weights without the 1×1 layers and **163,696** with them — **58% fewer**. The 5×5 branch on all 192 input channels is the expensive part; shrinking to 16 channels first makes it cheap. GoogLeNet stacked 9 such blocks (22 layers) with fewer weights than the much shallower AlexNet.

---
# A7 · ResNet (2015): skip connections
## The residual idea, by hand
Deeper networks should be at least as good as shallow ones: the extra layers could simply do **nothing** (pass their input on). In practice, plain networks find "do nothing" surprisingly hard to learn.

A plain block computes $y = F(x)$. A **residual** block computes $y = F(x) + x$: the input is added back (the **skip connection**).
- If the best thing a block can do is **nothing**, a residual block just sets $F(x) = 0$ — easy. A plain block must learn the identity exactly — hard.
- On the way back, the slope of "$+\,x$" is exactly 1: every block passes the incoming gradient straight through, plus its own part.

In [ ]:
x_small = np.array([[1.0, -2.0, 0.5]])
W_zero = np.zeros((3, 3))
F = lambda x, W: np.maximum(0, x @ W)                             # a tiny "block": dense + ReLU
print("plain block with all-zero weights   :", F(x_small, W_zero), " <- the input is lost")
print("residual block with all-zero weights:", F(x_small, W_zero) + x_small, " <- the input passes through")

## Why depth needs skip connections: 30-layer networks on MNIST
Slightly more than a toy: 30 dense layers of 64 neurons on MNIST digits, **plain** vs **residual** (each layer's output added to its input). First, before any training: how big is the gradient that reaches each layer? With sigmoid (the activation of early deep networks) and with ReLU:

In [ ]:
def deep_stack(depth, residual, activation, batchnorm=False, width=64):
    inputs = keras.Input((784,))
    h = layers.Dense(width, activation=activation)(inputs)
    for _ in range(depth):
        z = layers.Dense(width)(h)
        z = layers.BatchNormalization()(z) if batchnorm else z
        z = layers.Activation(activation)(z)
        h = layers.Add()([h, z]) if residual else z                 # the skip connection
    return keras.Model(inputs, layers.Dense(10)(h))

flat_train, flat_test = mx_train[:20000].reshape(-1, 784), mx_test.reshape(-1, 784)
labels_train = my_train[:20000]

_, ax = plt.subplots(figsize=(9, 4))
print(f"{'network':>26} | gradient size at layer 1 | at layer 15 | at layer 30")
print("-" * 82)
for activation in ["sigmoid", "relu"]:
    for residual in [False, True]:
        keras.utils.set_random_seed(0)
        model = deep_stack(30, residual, activation)
        with tf.GradientTape() as tape:
            loss = LOSS(labels_train[:256], model(flat_train[:256]))
        kernels = [w for w in model.trainable_weights if len(w.shape) == 2][1:-1]     # the 30 hidden layers
        sizes = [float(tf.norm(g)) for g in tape.gradient(loss, kernels)]
        name = f"{activation}, {'residual' if residual else 'plain'}"
        print(f"{name:>26} | {sizes[0]:>24.2e} | {sizes[14]:>11.2e} | {sizes[-1]:>11.2e}")
        ax.semilogy(range(1, 31), sizes, "o-" if residual else "s--", label=name)
ax.set(xlabel="layer (1 = nearest the input)", ylabel="gradient size (log scale)", title="Gradient reaching each of 30 layers, before training")
ax.legend()
plt.show()

**Our run (before any training):**

| network | layer 1 (input side) | layer 15 | layer 30 (output side) |
|---|---|---|---|
| sigmoid, plain | **0** (too small to store) | 0.00000000007 | 0.18 |
| sigmoid, residual | 0.9 | 3.5 | 7.8 |
| ReLU, plain | 0.000004 | 0.000004 | 0.000005 |
| ReLU, residual | 25,600 | 37,800 | 175,000 |

- **Sigmoid, plain:** the gradient shrinks at **every** layer on the way back (the slope of a sigmoid is at most ¼). After 30 layers nothing reaches the first layers — they cannot learn. This is the **vanishing gradient**.
- **Sigmoid, residual:** the "+ x" path carries the gradient straight back: every layer gets a usable gradient.
- **ReLU** behaves differently, and honestly: the plain stack's gradients are tiny but **even** across layers, and the residual stack **explodes** — every block adds its output on top of its input, so the values (and gradients) grow with depth. That is why real ResNet blocks always contain **BatchNorm**, which keeps the values in range.

Now train them (20,000 digits, 3 epochs). Plain vs residual with sigmoid, and with the **modern recipe** — ReLU **plus BatchNorm** (as in real ResNets):

In [ ]:
print(f"{'network (30 layers)':>32} | {'train accuracy by epoch':>24} | {'test':>6} | {'time':>5}")
print("-" * 80)
for activation, batchnorm in [("sigmoid", False), ("relu", True)]:
    for residual in [False, True]:
        keras.utils.set_random_seed(0)
        model = deep_stack(30, residual, activation, batchnorm)
        model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
        start = time.time()
        h = model.fit(flat_train, labels_train, batch_size=64, epochs=3, verbose=0).history
        name = f"{activation}{' + BatchNorm' if batchnorm else ''}, {'residual' if residual else 'plain'}"
        print(f"{name:>32} | {'  '.join(f'{100 * a:5.1f}%' for a in h['accuracy']):>24} | "
              f"{100 * model.evaluate(flat_test, my_test, verbose=0)[1]:5.1f}% | {time.time() - start:4.0f}s")

**Our run (30 layers, 3 epochs):**

| network | training accuracy after epochs 1 / 2 / 3 | test |
|---|---|---|
| sigmoid, plain | 10.5% / 10.4% / 10.5% | **9.8%** (guessing) |
| sigmoid, residual | 70.4% / 88.7% / 91.2% | **90.9%** |
| ReLU + BatchNorm, plain | 12.1% / 20.4% / 30.6% | **37.4%** |
| ReLU + BatchNorm, residual | 80.6% / 92.5% / 95.3% | **91.6%** |

The plain sigmoid stack learns **nothing** — exactly what the zero gradient predicted. More striking: even with the modern recipe (ReLU + BatchNorm), the plain 30-layer network crawls to 37%, while the **identical** network with skip connections reaches 92%. A simple 2-layer network reaches about 97% on MNIST (CNN-from-scratch notebook): the plain deep network is not limited by what it **can** represent, but by how hard it is to **train**. Skip connections fix the training.

## A residual **conv** block, by hand
A real ResNet block on feature maps: conv 3×3 ➜ ReLU ➜ conv 3×3, then **add the input**, then ReLU. When the block changes the size (stride 2) or the number of channels, $x$ no longer fits — the shortcut then uses a **1×1 conv with stride 2** to match (A4).

By hand in NumPy (the `conv_layer` of the convolution-basics notebook), then the same weights in Keras:

In [ ]:
def conv_layer(X, filters, padding=0, stride=1):
    # X (H, W, C_in), filters (k, k, C_in, C_out) -> (H', W', C_out)
    k = filters.shape[0]
    if padding:
        X = np.pad(X, ((padding, padding), (padding, padding), (0, 0)))
    windows = np.lib.stride_tricks.sliding_window_view(X, (k, k, X.shape[2]))[::stride, ::stride, 0]
    return np.einsum("ijklm,klmn->ijn", windows, filters)

rng = np.random.default_rng(3)
X = rng.normal(0, 1, (8, 8, 4)).astype("float32")                 # 8×8 feature maps, 4 channels
relu = lambda v: np.maximum(0, v)

# block 1: same size, same channels ➜ add x directly
W1, W2 = rng.normal(0, 0.3, (3, 3, 4, 4)).astype("float32"), rng.normal(0, 0.3, (3, 3, 4, 4)).astype("float32")
F1 = conv_layer(relu(conv_layer(X, W1, padding=1)), W2, padding=1)
same = relu(F1 + X)

# block 2: stride 2 and 4 ➜ 8 channels ➜ the shortcut needs a 1×1 conv with stride 2
V1, V2 = rng.normal(0, 0.3, (3, 3, 4, 8)).astype("float32"), rng.normal(0, 0.3, (3, 3, 8, 8)).astype("float32")
P = rng.normal(0, 0.3, (1, 1, 4, 8)).astype("float32")
F2 = conv_layer(relu(conv_layer(X, V1, padding=1, stride=2)), V2, padding=1)
shortcut = conv_layer(X, P, stride=2)
down = relu(F2 + shortcut)
print("input", X.shape, "| block 1 (identity shortcut) ➜", same.shape, "| block 2 (1×1 stride-2 shortcut) ➜", down.shape)

def keras_block(filters, stride, project):
    inputs = keras.Input((8, 8, 4))
    y = layers.ZeroPadding2D(1)(inputs)                            # pad 1 on every side, like our hand version
    y = layers.Conv2D(filters, 3, strides=stride, use_bias=False, activation="relu")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    s = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(inputs) if project else inputs
    return keras.Model(inputs, layers.ReLU()(layers.Add()([y, s])))

k1 = keras_block(4, 1, project=False)
k1.set_weights([W1, W2])
k2 = keras_block(8, 2, project=True)
k2.set_weights([V1, V2, P])
print("block 1: Keras == by hand?", np.allclose(k1(X[None])[0], same, atol=1e-4))
print("block 2: Keras == by hand?", np.allclose(k2(X[None])[0], down, atol=1e-4))

The block is ordinary convolutions plus **one addition**. That single `Add()` is the whole difference between a plain network and a ResNet.

## Plain vs residual CNNs on CIFAR-10 — **GPU recommended**
**On a CPU this section takes about 40 minutes** (5 networks × 10 epochs). Everything else in this notebook runs in a few minutes; skip these two cells on a CPU if time is short — our results are in the text below them.

Same design, more layers: stages of 16 ➜ 32 ➜ 64 filters, each stage `n` blocks of two 3×3 convolutions with BatchNorm, then GAP. Depth = 1 + 6n + 1 layers with weights: n = 1 ➜ **8 layers**, n = 3 ➜ **20**, n = 5 ➜ **32**. The residual versions are identical except for the `Add()` (and a 1×1 shortcut where the size changes).

10% of the training photos are held back for **validation**. Validation accuracy of deep BatchNorm networks jumps around from epoch to epoch on small data, so each network keeps the weights of its **best validation epoch** (the early-stopping idea of the CIFAR notebook) — and only those weights are tested, **once**.

In [ ]:
def conv_bn(x, filters, stride):
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    return layers.BatchNormalization()(x)

def build_net(n, residual):
    inputs = keras.Input((32, 32, 3))
    x = layers.ReLU()(conv_bn(inputs, 16, 1))
    for stage, filters in enumerate([16, 32, 64]):
        for block in range(n):
            stride = 2 if (stage > 0 and block == 0) else 1           # halve the size at the start of stages 2 and 3
            y = layers.ReLU()(conv_bn(x, filters, stride))
            y = conv_bn(y, filters, 1)
            if residual:                                              # the skip connection: add the block's input back
                shortcut = x if (stride == 1 and x.shape[-1] == filters) else layers.Conv2D(filters, 1, strides=stride, use_bias=False)(x)
                y = layers.Add()([y, shortcut])
            x = layers.ReLU()(y)
    outputs = layers.Dense(10)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inputs, outputs)

runs = {}

class KeepBest(keras.callbacks.Callback):
    # remember the weights of the epoch with the best validation accuracy, put them back at the end
    def on_train_begin(self, logs=None):
        self.best, self.best_epoch, self.best_weights = -1.0, 0, None
    def on_epoch_end(self, epoch, logs=None):
        if logs["val_accuracy"] > self.best:
            self.best, self.best_epoch, self.best_weights = logs["val_accuracy"], epoch + 1, self.model.get_weights()
    def on_train_end(self, logs=None):
        self.model.set_weights(self.best_weights)

def train(name, model):
    keras.utils.set_random_seed(42)
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
    keep = KeepBest()
    start = time.time()
    hist = model.fit(x_train, y_train, validation_split=0.1, batch_size=64, epochs=EPOCHS, callbacks=[keep], verbose=0).history
    test = 100 * model.evaluate(x_test, y_test, verbose=0)[1]
    runs[name] = {"hist": hist, "test": test, "weights": model.count_params(), "best_epoch": keep.best_epoch}
    print(f"{name:>10} | weights {model.count_params():>7,} | final train {100 * hist['accuracy'][-1]:5.1f}% | "
          f"best validation {100 * keep.best:5.1f}% (epoch {keep.best_epoch:>2}) | test {test:5.1f}% | {time.time() - start:4.0f} s")

def plot_runs(names, title):
    _, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    for name in names:
        axes[0].plot(range(1, EPOCHS + 1), [100 * a for a in runs[name]["hist"]["accuracy"]], "o-", label=name)
        axes[1].plot(range(1, EPOCHS + 1), [100 * a for a in runs[name]["hist"]["val_accuracy"]], "o-", label=name)
    axes[0].set(xlabel="epoch", ylabel="TRAINING accuracy (%)", title=title)
    axes[1].set(xlabel="epoch", ylabel="validation accuracy (%)", title="validation accuracy")
    axes[0].legend()
    axes[1].legend()
    plt.show()

for name, n in [("plain-8", 1), ("plain-20", 3), ("plain-32", 5)]:
    train(name, build_net(n, residual=False))
plot_runs(["plain-8", "plain-20", "plain-32"], "Plain CNNs: does more depth help?")

**Our run (training accuracy after 10 epochs):** plain-8 **72.2%**, plain-20 69.9%, plain-32 **53.9%**. The 32-layer network is worse on the **training photos themselves** — so this is not overfitting (that would mean *high* training accuracy). It could in theory copy the 8-layer network and let its extra layers do nothing; it simply fails to learn as well. This is the **degradation problem** that ResNet set out to fix.

Test accuracy (best-validation weights): plain-8 51.3%, plain-20 49.8%, plain-32 40.8% — the same order. With only 9,000 training photos and no augmentation, all of these networks overfit, so test accuracies around 50% are expected here (the CIFAR notebook reached 70% with augmentation and dropout).

In [ ]:
for name, n in [("resnet-20", 3), ("resnet-32", 5)]:
    train(name, build_net(n, residual=True))
plot_runs(["plain-20", "plain-32", "resnet-20", "resnet-32"], "Same depths: plain vs residual")

print(f"\n{'depth':>6} | {'plain train':>11} | {'residual train':>14} | {'plain test':>10} | {'residual test':>13}")
print("-" * 66)
for depth in [20, 32]:
    p, r = runs[f"plain-{depth}"], runs[f"resnet-{depth}"]
    print(f"{depth:>6} | {100 * p['hist']['accuracy'][-1]:10.1f}% | {100 * r['hist']['accuracy'][-1]:13.1f}% | {p['test']:9.1f}% | {r['test']:12.1f}%")

**Our run:**

| depth | plain: train / test | residual: train / test |
|---|---|---|
| 20 layers | 69.9% / 49.8% | **83.5%** / 50.7% |
| 32 layers | 53.9% / 40.8% | **79.9%** / **46.4%** |

- **Training:** with skip connections both depths train far better (+14 and +26 points), and the 32-layer network is no longer much worse than the 20-layer one. Depth stops hurting — that is what let ResNet go to 152 layers.
- **Test:** the gain is clear at 32 layers (+5.6 points) and small at 20 (+0.9). On 9,000 photos without augmentation, better training mostly turns into more overfitting; ResNet's advantage on new photos shows with more data, augmentation and more epochs (try it on a GPU with all 50,000 photos).
- **Why we kept the best-validation epoch:** the validation curves above jump by up to 20 points from one epoch to the next. In an earlier run that simply tested the **last** epoch, plain-20 scored 27.9% and resnet-20 39.0% — numbers that reflected the bad luck of the final epoch, not the networks.

---
# A8 · DenseNet (2017): connect every layer to every later layer
ResNet **adds** a block's input to its output. DenseNet **concatenates** instead: every layer receives the feature maps of **all** earlier layers in its block, stacked as channels, and adds a few new ones (the **growth rate**).

The difference on two tiny "feature maps" of 2 values each:

In [ ]:
a, b = np.array([1.0, 2.0]), np.array([3.0, 4.0])
print("ResNet   (add)        :", a + b, " <- the two are mixed into one; the original values are gone")
print("DenseNet (concatenate):", np.concatenate([a, b]), " <- both are kept, side by side")

A dense block in Keras: 16 input channels, 4 layers, growth rate 12 — each layer is BatchNorm ➜ ReLU ➜ 3×3 conv with **12** filters, and its output is concatenated to everything before it:

In [ ]:
inputs = keras.Input((32, 32, 16))
x = inputs
print(f"block input: {x.shape[-1]} channels")
for i in range(4):
    new = layers.BatchNormalization()(x)
    new = layers.ReLU()(new)
    new = layers.Conv2D(12, 3, padding="same")(new)                  # only 12 new feature maps …
    x = layers.Concatenate()([x, new])                               # … added to ALL earlier ones
    print(f"after layer {i + 1}: {x.shape[-1] - 12} kept + 12 new = {x.shape[-1]} channels")
dense_block = keras.Model(inputs, x)
print(f"dense block weights: {dense_block.count_params():,}")

Each layer is **narrow** (12 filters) because it can reuse everything earlier layers found — nothing has to be learned twice. That makes DenseNet small for its accuracy: DenseNet-121 has **8 M** weights against ResNet-50's **25.6 M**, with about the same ImageNet accuracy (A12). The cost: all those concatenated maps use a lot of memory while training. Pretrained in Keras: `keras.applications.DenseNet121 / 169 / 201`.

---
# A9 · MobileNet (2017): depthwise-separable convolution — split one convolution in two
1. **Depthwise**: one 3×3 filter **per channel** (no mixing between channels)
2. **Pointwise**: a 1×1 convolution mixes the channels

In [ ]:
inputs = keras.Input((32, 32, 128))
standard = keras.Model(inputs, layers.Conv2D(128, 3, padding="same")(inputs))
separable = keras.Model(inputs, layers.Conv2D(128, 1)(layers.DepthwiseConv2D(3, padding="same")(inputs)))
x = np.random.default_rng(2).random((1, 32, 32, 128)).astype("float32")
print(f"standard 3×3 conv 128➜128 : {standard.count_params():>7,} weights, output {tuple(standard(x).shape)}")
print(f"depthwise 3×3 + 1×1        : {separable.count_params():>7,} weights, output {tuple(separable(x).shape)}")
print(f"{standard.count_params() / separable.count_params():.1f}× fewer weights for the same output shape — why MobileNet runs on phones")

---
# A10 · EfficientNet (2019): scale depth, width and image size together
To make a CNN more accurate you can make it
- **deeper** (more layers) — sees bigger patterns,
- **wider** (more filters per layer) — finds more kinds of details,
- or feed it **bigger images** — more detail to see.

Earlier networks grew **one** of these at a time. EfficientNet's finding: grow **all three together**, in a fixed ratio, from one small, carefully designed starting network (**B0**, built from MobileNet-style blocks). Each step up (B1, B2, … B7) makes it roughly 1.2× deeper, 1.1× wider and uses about 1.15× larger images. The family, built here without downloading weights:

In [ ]:
EFFICIENT = [("B0", keras.applications.EfficientNetB0, 77.1), ("B1", keras.applications.EfficientNetB1, 79.1),
             ("B2", keras.applications.EfficientNetB2, 80.1), ("B3", keras.applications.EfficientNetB3, 81.6),
             ("B4", keras.applications.EfficientNetB4, 82.9), ("B5", keras.applications.EfficientNetB5, 83.6),
             ("B6", keras.applications.EfficientNetB6, 84.0), ("B7", keras.applications.EfficientNetB7, 84.3)]
print(f"{'model':>6} | {'image size':>10} | {'layers':>6} | {'weights':>11} | {'published ImageNet top-1':>24}")
print("-" * 70)
for name, build, top1 in EFFICIENT:
    model = build(weights=None)
    size = model.input_shape[1]
    print(f"{name:>6} | {f'{size}×{size}':>10} | {len(model.layers):>6} | {model.count_params():>11,} | {top1:>23.1f}%")
    del model

__EFFICIENT__

---
# A11 · ConvNeXt (2022): a CNN modernised with ideas from transformers
Around 2020, **Vision Transformers** (Transformers folder) started beating CNNs. ConvNeXt asked: is it the transformer, or the **training tricks and design choices** that came with it? It took a ResNet and changed it step by step:

| ResNet block | ConvNeXt block |
|---|---|
| 3×3 convolutions | **7×7 depthwise** convolution (each channel on its own, as in MobileNet) — sees more at once, cheaply |
| BatchNorm | **LayerNorm** (normalise each pixel's channels, as in transformers) |
| ReLU after every conv | **one GELU** per block (a smooth ReLU) |
| bottleneck: wide ➜ narrow ➜ wide | **inverted**: narrow ➜ 4× wider ➜ narrow (as in transformers' MLP) |

The result is a **pure CNN** that matches transformers of the same size. One ConvNeXt block in Keras:

In [ ]:
inputs = keras.Input((56, 56, 96))
x = layers.DepthwiseConv2D(7, padding="same")(inputs)               # 7×7, one filter per channel
x = layers.LayerNormalization(epsilon=1e-6)(x)
x = layers.Dense(4 * 96)(x)                                         # a Dense on the last axis = a 1×1 conv: 96 ➜ 384
x = layers.Activation("gelu")(x)
x = layers.Dense(96)(x)                                             # 384 ➜ 96
outputs = layers.Add()([inputs, x])                                 # ResNet's skip connection is still there
convnext_block = keras.Model(inputs, outputs)
print(f"one ConvNeXt block (96 channels): {convnext_block.count_params():,} weights | output {convnext_block.output.shape[1:]}")

tiny = keras.applications.ConvNeXtTiny(weights=None)
print(f"ConvNeXtTiny: {tiny.count_params():,} weights, {len(tiny.layers)} layers (pretrained in Keras: ConvNeXtTiny / Small / Base / Large / XLarge)")
del tiny

---
# A12 · Six pretrained models, head to head
Keras includes these networks **already trained on ImageNet** (1,000 classes — including about 120 dog breeds). Download six (about 400 MB together) and run them, as they are, on the same photos. For each: top-3 answers, size, and time per photo on **this** machine. (VGG16 is also available but weighs 528 MB, so we only list its size.)

The last photo is a 32×32 CIFAR test photo blown up to 224×224 — what happens with a tiny, blurry input?

In [ ]:
PHOTOS = {
    "dog":      "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg",
    "cat":      "https://storage.googleapis.com/download.tensorflow.org/example_images/320px-Felis_catus-cat_on_snow.jpg",
    "portrait": "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg",
}
photo_paths = {name: keras.utils.get_file(url.split("/")[-1], url) for name, url in PHOTOS.items()}
ship = x_test[np.where(y_test == 8)[0][0]]                          # a CIFAR ship, 32×32

def photos_at(size):
    imgs = {name: np.array(keras.utils.load_img(p, target_size=(size, size)), dtype="float32") for name, p in photo_paths.items()}
    imgs["CIFAR ship (32×32)"] = tf.image.resize(ship * 255, (size, size)).numpy()
    return imgs

apps = keras.applications
MODELS = {   # name: (builder, its module (preprocessing + class names), input size, published ImageNet top-1 %)
    "MobileNetV2":    (apps.MobileNetV2, apps.mobilenet_v2, 224, 71.3),
    "EfficientNetB0": (apps.EfficientNetB0, apps.efficientnet, 224, 77.1),
    "ResNet50":       (apps.ResNet50, apps.resnet50, 224, 74.9),
    "InceptionV3":    (apps.InceptionV3, apps.inception_v3, 299, 77.9),
    "DenseNet121":    (apps.DenseNet121, apps.densenet, 224, 75.0),
    "ConvNeXtTiny":   (apps.ConvNeXtTiny, apps.convnext, 224, 81.3),
}
answers, speed = {}, {}
for model_name, (build, module, size, _) in MODELS.items():
    model = build(weights="imagenet")
    imgs = photos_at(size)
    for name, img in imgs.items():
        probs = model(module.preprocess_input(img[None].copy()), training=False).numpy()
        answers[(model_name, name)] = [(label, 100 * p) for _, label, p in module.decode_predictions(probs, top=3)[0]]
    batch = module.preprocess_input(imgs["dog"][None].copy())
    for _ in range(3):
        model(batch, training=False)                                # warm up
    start = time.time()
    for _ in range(10):
        model(batch, training=False)
    speed[model_name] = {"ms": 100 * (time.time() - start), "MB": model.count_params() * 4 / 1e6, "weights": model.count_params(), "layers": len(model.layers)}
    print(f"{model_name} done")
    del model

_, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, img) in zip(axes, photos_at(224).items()):
    ax.imshow(img.astype("uint8"))
    ax.set(title=name, xticks=[], yticks=[])
plt.show()
for name in photos_at(32):
    print(f"\n{name}")
    for model_name in MODELS:
        print(f"   {model_name:>14}: " + " | ".join(f"{label} {p:.0f}%" for label, p in answers[(model_name, name)]))

In [ ]:
print(f"{'model':>15} | {'year':>4} | {'layers':>6} | {'weights':>11} | {'size':>6} | {'one photo, this machine':>23} | {'published top-1':>15}")
print("-" * 100)
years = {"MobileNetV2": 2018, "EfficientNetB0": 2019, "ResNet50": 2015, "InceptionV3": 2015, "DenseNet121": 2017, "ConvNeXtTiny": 2022}
for model_name, s in speed.items():
    print(f"{model_name:>15} | {years[model_name]} | {s['layers']:>6} | {s['weights']:>11,} | {s['MB']:>3.0f} MB | {s['ms']:>20.0f} ms | {MODELS[model_name][3]:>14.1f}%")
vgg = apps.VGG16(weights=None)
print(f"{'VGG16':>15} | 2014 | {len(vgg.layers):>6} | {vgg.count_params():>11,} | {vgg.count_params() * 4 / 1e6:>3.0f} MB | {'(not downloaded)':>23} | {71.3:>14.1f}%")
del vgg

__SHOWDOWN__

---
# Part B — Detection architectures
Detection = **what** and **where** (the object-detection-from-scratch notebook). How do the big detectors find many objects of 80 kinds in a real photo?

# B1 · The naive way: sliding windows
Take a good **classifier** and run it on every window of the photo, at several window sizes. It works — but count the windows:

In [ ]:
W_img, H_img, STRIDE = 640, 480, 16
total = 0
for win in [64, 128, 256]:
    n = ((W_img - win) // STRIDE + 1) * ((H_img - win) // STRIDE + 1)
    total += n
    print(f"window {win:>3}×{win:<3} moved {STRIDE} pixels at a time: {n:>5,} windows")
print(f"total: {total:,} windows ➜ {total:,} classifier runs for ONE photo")
print(f"at ~{speed['MobileNetV2']['ms']:.0f} ms per run (MobileNetV2 on this machine, A12): about {total * speed['MobileNetV2']['ms'] / 1000 / 60:.0f} minutes per photo")

Far too slow — and it still misses objects whose shape is not square. Every detector below avoids running the classifier thousands of times.

---
# B2 · Two-stage detectors: R-CNN ➜ Fast R-CNN ➜ Faster R-CNN
**Stage 1** proposes about 2,000 **regions** that *might* contain an object ("something is here"); **stage 2** classifies each region and refines its box.
- **R-CNN (2014):** regions from a classical algorithm, each region cropped and sent through a CNN separately — accurate, but about a minute per photo.
- **Fast R-CNN (2015):** run the CNN **once** on the whole photo, then cut every region out of the **feature map** (RoI pooling) — much faster.
- **Faster R-CNN (2015):** the regions also come from the network: a small **Region Proposal Network** slides over the feature map and, at every position, scores a fixed set of **anchor boxes**.

**Anchors** are preset boxes of a few sizes and shapes placed at every position of the feature map; the network only has to say *"object or not"* for each anchor and *how to nudge it* to fit. Three sizes × three shapes = 9 anchors per position:

In [ ]:
beach_path = keras.utils.get_file("od_image2.jpg", "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image2.jpg")
beach = np.array(keras.utils.load_img(beach_path))
H_b, W_b = beach.shape[:2]
cx, cy = W_b * 0.55, H_b * 0.45                                     # one position on the photo
_, ax = plt.subplots(figsize=(10, 6.5))
ax.imshow(beach)
colours = {128: "yellow", 256: "orange", 512: "red"}
for size in [128, 256, 512]:
    for ratio in [0.5, 1.0, 2.0]:                                   # wide, square, tall
        w, h = size * np.sqrt(1 / ratio), size * np.sqrt(ratio)
        ax.add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, edgecolor=colours[size], lw=1.5))
ax.plot(cx, cy, "c+", markersize=15, mew=3)
ax.set(title="9 anchor boxes at ONE position: 3 sizes (yellow, orange, red) × 3 shapes", xticks=[], yticks=[])
plt.show()
stride = 16
n_positions = (H_b // stride) * (W_b // stride)
print(f"photo {W_b}×{H_b}, feature map every {stride} pixels: {n_positions:,} positions × 9 anchors = {9 * n_positions:,} anchors, scored in ONE pass")

---
# B3 · One-stage detectors: YOLO and SSD
Skip the proposals: predict boxes and classes **directly**, in one pass, from a grid — exactly the **mini-YOLO** of the detection-from-scratch notebook, grown up.
- **YOLO (2016):** one grid; each cell predicts boxes + objectness + class. Very fast — real-time video.
- **SSD (2016):** predictions from **several** feature maps of different sizes: the fine maps (early layers, many cells) catch **small** objects, the coarse maps (deep layers, few cells) catch **big** ones. Each cell uses anchors, as in Faster R-CNN.

In [ ]:
_, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, cells_ in zip(axes, [32, 16, 8]):
    ax.imshow(beach)
    for k in range(1, cells_):
        ax.axhline(k * H_b / cells_, color="white", lw=0.4)
        ax.axvline(k * W_b / cells_, color="white", lw=0.4)
    ax.set(title=f"{cells_}×{cells_} grid: {'small' if cells_ == 32 else 'medium' if cells_ == 16 else 'big'} objects", xticks=[], yticks=[])
plt.suptitle("SSD / feature pyramid idea: the same photo seen at three grid sizes")
plt.show()

---
# B4 · RetinaNet (2017): the focal loss
One-stage detectors score **tens of thousands** of anchors per photo, and almost all of them are plain background. Each of those is **easy** (the model is already 99% sure "no object"), but there are so many that together they swamp the few hard, interesting anchors.

**Focal loss** fixes this by multiplying the usual cross-entropy by $(1 - p)^\gamma$, where $p$ is the probability the model gives to the **right** answer:
$$\text{CE} = -\log p \qquad\qquad \text{FL} = -(1 - p)^{\gamma}\,\log p \qquad (\gamma = 2)$$

In [ ]:
print(f"{'p (right answer)':>16} | {'cross-entropy':>13} | {'focal loss':>10} | {'focal / CE':>10}")
print("-" * 60)
for p in [0.99, 0.9, 0.5, 0.1]:
    ce, fl = -np.log(p), -(1 - p) ** 2 * np.log(p)
    kind = "easy background" if p >= 0.9 else "hard"
    print(f"{p:>16} | {ce:>13.4f} | {fl:>10.6f} | {fl / ce:>9.4f}×   ({kind})")

p = np.linspace(0.01, 1, 200)
plt.figure(figsize=(7, 3.8))
plt.plot(p, -np.log(p), label="cross-entropy")
for g in [1, 2, 5]:
    plt.plot(p, -(1 - p) ** g * np.log(p), label=f"focal loss, γ = {g}")
plt.xlabel("probability given to the right answer")
plt.ylabel("loss")
plt.title("Focal loss: easy examples (right side) count almost nothing")
plt.legend()
plt.show()

An anchor the model already gets right with 99% counts **10,000× less** than with ordinary cross-entropy; a hard one (10%) keeps most of its loss. RetinaNet = this focal loss + a **feature pyramid** (the SSD idea, with deep features passed back up to the fine maps) on a ResNet backbone — a one-stage detector as accurate as two-stage ones.

---
# B5 · Two pretrained detectors, head to head
Both were trained on **COCO** (330,000 photos, **80 kinds of objects**: person, car, dog, kite, …):
- **RetinaNet** (one-stage, ResNet-50 + feature pyramid) from **KerasHub**
- **Faster R-CNN** (two-stage, ResNet-50) from **TensorFlow Hub** (`tensorflow_hub` is preinstalled in Colab; elsewhere `pip install tensorflow-hub`)

Both use the original COCO class numbers (1 = person, … 90 = toothbrush):

In [ ]:
COCO = {1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane", 6: "bus", 7: "train", 8: "truck", 9: "boat",
        10: "traffic light", 11: "fire hydrant", 13: "stop sign", 14: "parking meter", 15: "bench", 16: "bird", 17: "cat",
        18: "dog", 19: "horse", 20: "sheep", 21: "cow", 22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe", 27: "backpack",
        28: "umbrella", 31: "handbag", 32: "tie", 33: "suitcase", 34: "frisbee", 35: "skis", 36: "snowboard", 37: "sports ball",
        38: "kite", 39: "baseball bat", 40: "baseball glove", 41: "skateboard", 42: "surfboard", 43: "tennis racket", 44: "bottle",
        46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon", 51: "bowl", 52: "banana", 53: "apple", 54: "sandwich",
        55: "orange", 56: "broccoli", 57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair", 63: "couch",
        64: "potted plant", 65: "bed", 67: "dining table", 70: "toilet", 72: "tv", 73: "laptop", 74: "mouse", 75: "remote",
        76: "keyboard", 77: "cell phone", 78: "microwave", 79: "oven", 80: "toaster", 81: "sink", 82: "refrigerator", 84: "book",
        85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear", 89: "hair drier", 90: "toothbrush"}

dogs_path = keras.utils.get_file("od_image1.jpg", "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image1.jpg")
SCENES = {"beach": beach_path, "dogs": dogs_path, "Labrador": photo_paths["dog"], "portrait": photo_paths["portrait"]}
scenes = {name: np.array(keras.utils.load_img(p)) for name, p in SCENES.items()}
for name, img in scenes.items():
    print(f"{name:>9}: {img.shape[1]}×{img.shape[0]}")

In [ ]:
import keras_hub
retinanet = keras_hub.models.ObjectDetector.from_preset("retinanet_resnet50_fpn_coco")
print(f"RetinaNet: {retinanet.count_params():,} weights")

def run_retinanet(img):
    out = retinanet.predict(img[None], verbose=0)
    n = int(out["num_detections"][0])
    boxes = out["boxes"][0][:n]                                     # (y_min, x_min, y_max, x_max) in pixels
    return [((b[1], b[0], b[3], b[2]), int(c), float(s)) for b, c, s in zip(boxes, out["labels"][0][:n], out["confidence"][0][:n])]

In [ ]:
import tensorflow_hub as hub
faster_rcnn = hub.load("https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1")

def run_faster_rcnn(img):
    out = faster_rcnn(tf.constant(img[None]))
    H, W = img.shape[:2]
    found = []
    for b, c, s in zip(out["detection_boxes"][0].numpy(), out["detection_classes"][0].numpy(), out["detection_scores"][0].numpy()):
        y0, x0, y1, x1 = b                                          # (y_min, x_min, y_max, x_max) as fractions of the photo
        found.append(((x0 * W, y0 * H, x1 * W, y1 * H), int(c), float(s)))
    return found

In [ ]:
THRESHOLD = 0.5
DETECTORS = {"RetinaNet (one-stage)": run_retinanet, "Faster R-CNN (two-stage)": run_faster_rcnn}
results_det, timing = {}, {}
for det_name, run in DETECTORS.items():
    run(scenes["beach"])                                            # first call builds the graph: not timed
    times = []
    for name, img in scenes.items():
        start = time.time()
        results_det[(det_name, name)] = [f for f in run(img) if f[2] >= THRESHOLD]
        times.append(time.time() - start)
    timing[det_name] = np.mean(times)

for name, img in scenes.items():
    _, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, det_name in zip(axes, DETECTORS):
        ax.imshow(img)
        for (x0, y0, x1, y1), c, s in results_det[(det_name, name)]:
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="lime", lw=2))
            ax.text(x0, y0 - 3, f"{COCO.get(c, c)} {100 * s:.0f}%", color="black", fontsize=8, backgroundcolor="lime")
        ax.set(title=f"{det_name}: {len(results_det[(det_name, name)])} objects ≥ {THRESHOLD:.0%}", xticks=[], yticks=[])
    plt.tight_layout()
    plt.show()

In [ ]:
print(f"{'photo':>9} | {'RetinaNet (one-stage)':>45} | {'Faster R-CNN (two-stage)':>45}")
print("-" * 106)
for name in scenes:
    cols = []
    for det_name in DETECTORS:
        counts = {}
        for _, c, _ in results_det[(det_name, name)]:
            counts[COCO.get(c, c)] = counts.get(COCO.get(c, c), 0) + 1
        cols.append(", ".join(f"{v} {k}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])) or "nothing")
    print(f"{name:>9} | {cols[0]:>45} | {cols[1]:>45}")
print()
for det_name in DETECTORS:
    print(f"{det_name}: {timing[det_name]:.1f} s per photo on this machine")

__DETECTORS__

---
# B6 · Which detector when?

| need | choose | why |
|---|---|---|
| real-time video, phones, cameras | **one-stage** (YOLO family, SSD-MobileNet) | one pass; small backbones |
| best accuracy, small or crowded objects, time is not critical | **two-stage** (Faster R-CNN and successors) or RetinaNet | proposals + refinement / focal loss |
| your own objects (number plates, products, defects) | a **pretrained** detector, **fine-tuned** on a few hundred labelled photos | transfer-learning notebook |

---
# Summary

**Classification architectures**

| architecture | year | the one idea | what we measured / showed | pretrained in Keras |
|---|---|---|---|---|
| LeNet-5 | 1998 | conv ➜ pool ➜ dense | 98% on digits, only 45% on CIFAR photos | no |
| AlexNet | 2012 | deep + ReLU + dropout + GPUs | __SUM_ALEX__ | no |
| Network-in-Network | 2013 | 1×1 conv + global average pooling | 1×1 = dense layer at every pixel; head 40,970 ➜ 650 weights | no |
| VGG | 2014 | only 3×3 filters, stacked | two 3×3 = one 5×5 view, 28% fewer weights | VGG16, VGG19 |
| GoogLeNet / Inception | 2014 | parallel branches + 1×1 bottlenecks | same output, 58% fewer weights | InceptionV3 |
| ResNet | 2015 | skip connections | 30-layer MNIST: plain 37% vs residual 92% | ResNet50/101/152 |
| DenseNet | 2017 | concatenate every earlier layer | channels grow 16 ➜ 64 with 12 new maps per layer | DenseNet121/169/201 |
| MobileNet | 2017 | depthwise-separable convolution | 8.3× fewer weights per conv | MobileNetV2/V3 |
| EfficientNet | 2019 | scale depth, width, image size together | __SUM_EFF__ | EfficientNetB0–B7 |
| ConvNeXt | 2022 | ResNet modernised (7×7 depthwise, LayerNorm, GELU) | __SUM_CONV__ | ConvNeXtTiny … XLarge |

**Detection architectures**

| family | examples | idea | measured here |
|---|---|---|---|
| sliding window | — | classifier on every window | __SUM_SLIDE__ |
| two-stage | R-CNN ➜ Fast ➜ Faster R-CNN | propose regions (anchors), then classify each | __SUM_FRCNN__ |
| one-stage | YOLO, SSD, RetinaNet | boxes + classes from grids in one pass; focal loss | __SUM_RETINA__ |

**Next notebook: transfer learning** — instead of using these pretrained networks as they are, **retrain** them for your own classes (pet breeds, flowers) and your own objects (number plates).